In [2]:
import pandas as pd

In [2]:
crime = pd.read_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Crime\Crime_2001_2023_Cleaned.csv")
scoio_economic = pd.read_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Socio_Economic\Socio_Economic_Final.csv")
weather = pd.read_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Weather\Weather_Final.csv")
mobility = pd.read_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset/Mobality/Mobality_Final.csv"
)


In [3]:
print(crime.shape)
print(scoio_economic.shape)
print(weather.shape)
print(mobility.shape)

(7970734, 18)
(154, 6)
(8400, 12)
(9190, 11)


Merge Crime and socio economic 

In [4]:
crime = crime.rename(columns={
    'Community Area': 'community_area',
    'Year': 'year'
})

In [5]:
# Period 1 → use 2012 socio data
period_1 = crime[crime['year'] <= 2012].copy()

# Period 2 → use 2020 socio data
period_2 = crime[crime['year'] >= 2013].copy()

In [6]:
socio_2012 = scoio_economic[scoio_economic['year'] == 2012].copy()
socio_2020 = scoio_economic[scoio_economic['year'] == 2020].copy()

In [7]:
print(period_1.shape)

(5126709, 18)


In [8]:
print(period_2.shape)

(2844025, 18)


In [9]:
# 5126709 + 1118311 + 1725714

In [10]:
socio_2012.shape
socio_2012.head(1)

,community_area,community_name,per_capita_income,unemployment_rate,no_highschool_pct,year
0,1,rogers park,23939,8.7,18.2,2012


In [11]:
socio_2020.shape
socio_2020.head(1)

,community_area,community_name,per_capita_income,unemployment_rate,no_highschool_pct,year
1,1,rogers park,28052,5.388607,8.396575,2020


In [12]:
socio_2012 = socio_2012.drop(columns=['year'])
socio_2020 = socio_2020.drop(columns=['year'])

In [13]:
merged_1 = period_1.merge(socio_2012, on='community_area', how='left')
merged_2 = period_2.merge(socio_2020, on='community_area', how='left')


In [14]:
final_merge = pd.concat([merged_1, merged_2], ignore_index=True)

final_merge.head(1)

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,community_area,...,Latitude,Longitude,month,day_of_week,hour,is_weekend,community_name,per_capita_income,unemployment_rate,no_highschool_pct
0,6255892,2008-05-17 18:00:00,ROBBERY,ARMED - HANDGUN,RESIDENCE,0,0,511,5.0,49,...,41.71004,-87.624796,5,5,18,1,roseland,17949.0,20.3,16.9


In [15]:
final_merge[['per_capita_income', 'unemployment_rate', 'no_highschool_pct']].isnull().sum()

per_capita_income    76
unemployment_rate    76
no_highschool_pct    76
dtype: int64

In [16]:
missing_rows = final_merge[final_merge['per_capita_income'].isnull()]

missing_rows['community_area'].value_counts()

community_area
0    76
Name: count, dtype: int64

In [17]:
final_merge = final_merge[final_merge['community_area'] != 0]


In [18]:
final_merge[['per_capita_income', 'unemployment_rate', 'no_highschool_pct']].isnull().sum()

per_capita_income    0
unemployment_rate    0
no_highschool_pct    0
dtype: int64

In [19]:
final_merge['community_area'].min(), final_merge['community_area'].max()

(1, 77)

In [20]:
final_merge.head()

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,community_area,...,Latitude,Longitude,month,day_of_week,hour,is_weekend,community_name,per_capita_income,unemployment_rate,no_highschool_pct
0,6255892,2008-05-17 18:00:00,ROBBERY,ARMED - HANDGUN,RESIDENCE,0,0,511,5.0,49,...,41.710040,-87.624796,5,5,18,1,roseland,17949.0,20.3,16.9
1,6272641,2008-05-27 01:00:00,ROBBERY,STRONG ARM - NO WEAPON,STREET,0,1,512,5.0,49,...,41.703007,-87.625785,5,1,1,0,roseland,17949.0,20.3,16.9
2,6438609,2008-08-05 22:37:00,ROBBERY,ARMED - HANDGUN,SCHOOL - PUBLIC GROUNDS,0,0,523,5.0,53,...,41.664425,-87.639053,8,1,22,0,west pullman,16563.0,19.4,20.5
3,6680276,2008-12-27 20:00:00,BURGLARY,FORCIBLE ENTRY,RESIDENCE - GARAGE,0,0,1622,16.0,11,...,41.987326,-87.770650,12,5,20,1,jefferson park,27751.0,12.4,13.4
4,13209443,2004-09-17 00:00:00,OFFENSE INVOLVING CHILDREN,AGGRAVATED CRIMINAL SEXUAL ABUSE BY FAMILY MEMBER,RESIDENCE,0,1,935,9.0,61,...,41.808292,-87.637870,9,4,0,0,new city,12765.0,23.0,41.5


Weather

In [21]:
weather['date'] = pd.to_datetime(weather['date'])

weather.head()


,Unnamed: 0,date,TMAX,TMIN,PRCP,SNOW,AWND,temp_avg,is_rain,is_snow,temp_category,high_wind
0,0,2001-01-01,24,5,0.0,0.0,7.61,14.5,0,0,cold,0
1,1,2001-01-02,19,5,0.0,0.0,8.50,12.0,0,0,cold,0
2,2,2001-01-03,28,7,0.0,0.0,11.41,17.5,0,0,cold,0
3,3,2001-01-04,30,19,0.0,0.0,11.63,24.5,0,0,cold,0
4,4,2001-01-05,36,21,0.0,0.0,13.87,28.5,0,0,cold,0


In [22]:
final_merge['date'] = pd.to_datetime(final_merge['Date']).dt.date

final_merge['date'] = pd.to_datetime(final_merge['date'])

final_merge[['Date', 'date']].head()

,Date,date
0,2008-05-17 18:00:00,2008-05-17
1,2008-05-27 01:00:00,2008-05-27
2,2008-08-05 22:37:00,2008-08-05
3,2008-12-27 20:00:00,2008-12-27
4,2004-09-17 00:00:00,2004-09-17


In [23]:
weather.columns

Index(['Unnamed: 0', 'date', 'TMAX', 'TMIN', 'PRCP', 'SNOW', 'AWND',
       'temp_avg', 'is_rain', 'is_snow', 'temp_category', 'high_wind'],
      dtype='object')

In [24]:
final_merge = final_merge.merge(
    weather[['date', 'TMAX', 'TMIN', 'PRCP', 'SNOW', 'AWND',
       'temp_avg', 'is_rain', 'is_snow', 'temp_category', 'high_wind']],
    on='date',
    how='left'
)

In [25]:
final_merge.shape

(7970658, 33)

Mobility

In [28]:
print(final_merge['date'].dtype)
print(mobility['date'].dtype)

datetime64[ns]
object


In [29]:
mobility['date'] = pd.to_datetime(mobility['date'])

In [30]:
print(final_merge['date'].dtype)
print(mobility['date'].dtype)

datetime64[ns]
datetime64[ns]


In [31]:
final_df = pd.merge(
    final_merge,
    mobility,
    on='date',
    how='left'
)

In [32]:
print("Final Dataset Shape :", final_df.shape)
print(final_df.head())

Final Dataset Shape : (7970658, 43)
         ID                 Date                Primary Type  \
0   6255892  2008-05-17 18:00:00                     ROBBERY   
1   6272641  2008-05-27 01:00:00                     ROBBERY   
2   6438609  2008-08-05 22:37:00                     ROBBERY   
3   6680276  2008-12-27 20:00:00                    BURGLARY   
4  13209443  2004-09-17 00:00:00  OFFENSE INVOLVING CHILDREN   

                                         Description     Location Description  \
0                                    ARMED - HANDGUN                RESIDENCE   
1                             STRONG ARM - NO WEAPON                   STREET   
2                                    ARMED - HANDGUN  SCHOOL - PUBLIC GROUNDS   
3                                     FORCIBLE ENTRY       RESIDENCE - GARAGE   
4  AGGRAVATED CRIMINAL SEXUAL ABUSE BY FAMILY MEMBER                RESIDENCE   

   Arrest  Domestic  Beat  District  community_area  ... service_date  \
0       0         0

In [33]:
final_df.to_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Crime_Socio_Economic_Weather_Mobility.csv", index=False)

In [34]:
crime_soc_eco_wea_mob = pd.read_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Crime_Socio_Economic_Weather_Mobility.csv")
crime_soc_eco_wea_mob.shape

(7970658, 43)

In [35]:
crime_soc_eco_wea_mob.head(1)

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,community_area,...,service_date,day_type,bus,rail_boardings,total_rides,year_y,month_y,day,day_of_week_y,is_weekend_y
0,6255892,2008-05-17 18:00:00,ROBBERY,ARMED - HANDGUN,RESIDENCE,0,0,511,5.0,49,...,2008-05-17,A,"749,048","350,884","1,099,932",2008,5,17,Saturday,1


In [36]:
crime_soc_eco_wea_mob.tail(1)

,ID,Date,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,community_area,...,service_date,day_type,bus,rail_boardings,total_rides,year_y,month_y,day,day_of_week_y,is_weekend_y
7970657,14016205,2021-10-07 09:30:00,BATTERY,DOMESTIC BATTERY SIMPLE,APARTMENT,0,1,1014,10.0,29,...,2021-10-07,W,"447,936","323,569","771,505",2021,10,7,Thursday,0


In [37]:
crime_soc_eco_wea_mob.isnull().sum()

ID                      0
Date                    0
Primary Type            0
Description             0
Location Description    0
Arrest                  0
Domestic                0
Beat                    0
District                0
community_area          0
FBI Code                0
year_x                  0
Latitude                0
Longitude               0
month_x                 0
day_of_week_x           0
hour                    0
is_weekend_x            0
community_name          0
per_capita_income       0
unemployment_rate       0
no_highschool_pct       0
date                    0
TMAX                    0
TMIN                    0
PRCP                    0
SNOW                    0
AWND                    0
temp_avg                0
is_rain                 0
is_snow                 0
temp_category           0
high_wind               0
service_date            0
day_type                0
bus                     0
rail_boardings          0
total_rides             0
year_y      

In [38]:
df = crime_soc_eco_wea_mob.copy()

In [39]:
print(df.columns)


Index(['ID', 'Date', 'Primary Type', 'Description', 'Location Description',
       'Arrest', 'Domestic', 'Beat', 'District', 'community_area', 'FBI Code',
       'year_x', 'Latitude', 'Longitude', 'month_x', 'day_of_week_x', 'hour',
       'is_weekend_x', 'community_name', 'per_capita_income',
       'unemployment_rate', 'no_highschool_pct', 'date', 'TMAX', 'TMIN',
       'PRCP', 'SNOW', 'AWND', 'temp_avg', 'is_rain', 'is_snow',
       'temp_category', 'high_wind', 'service_date', 'day_type', 'bus',
       'rail_boardings', 'total_rides', 'year_y', 'month_y', 'day',
       'day_of_week_y', 'is_weekend_y'],
      dtype='object')


In [40]:
(df['year_x'] == df['year_y']).all()


np.True_

In [41]:
df = df.drop(columns=[
    'year_y',
    'month_y',
    'day_of_week_y',
    'is_weekend_y'
])

df = df.rename(columns={
    'year_x': 'year',
    'month_x': 'month',
    'day_of_week_x': 'day_of_week',
    'is_weekend_x': 'is_weekend'
})

In [42]:
print(df.columns)


Index(['ID', 'Date', 'Primary Type', 'Description', 'Location Description',
       'Arrest', 'Domestic', 'Beat', 'District', 'community_area', 'FBI Code',
       'year', 'Latitude', 'Longitude', 'month', 'day_of_week', 'hour',
       'is_weekend', 'community_name', 'per_capita_income',
       'unemployment_rate', 'no_highschool_pct', 'date', 'TMAX', 'TMIN',
       'PRCP', 'SNOW', 'AWND', 'temp_avg', 'is_rain', 'is_snow',
       'temp_category', 'high_wind', 'service_date', 'day_type', 'bus',
       'rail_boardings', 'total_rides', 'day'],
      dtype='object')


In [43]:
df.to_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Crime_Socio_Economic_Weather_Mobility.csv", index=False)

In [45]:
print(final_merge.shape)
print(final_df.shape)
print(df.shape)

(7970658, 33)
(7970658, 43)
(7970658, 39)


# Making it directly training ready

In [3]:
df = pd.read_csv(r'../1.0.Dataset/Crime_Socio_Economic_Weather_Mobility.csv')


In [5]:
df.shape

(7970658, 39)

In [6]:
df.dtypes

ID                        int64
Date                     object
Primary Type             object
Description              object
Location Description     object
Arrest                    int64
Domestic                  int64
Beat                      int64
District                float64
community_area            int64
FBI Code                 object
year                      int64
Latitude                float64
Longitude               float64
month                     int64
day_of_week               int64
hour                      int64
is_weekend                int64
community_name           object
per_capita_income       float64
unemployment_rate       float64
no_highschool_pct       float64
date                     object
TMAX                      int64
TMIN                      int64
PRCP                    float64
SNOW                    float64
AWND                    float64
temp_avg                float64
is_rain                   int64
is_snow                   int64
temp_cat

In [7]:
mobility_cols = ['bus', 'rail_boardings', 'total_rides']

for col in mobility_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(',', '', regex=False)
    )

    df[col] = pd.to_numeric(df[col], errors='coerce')

In [8]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')
df['date'] = df['date'].map(pd.Timestamp.toordinal)


In [9]:
df.dtypes


ID                        int64
Date                     object
Primary Type             object
Description              object
Location Description     object
Arrest                    int64
Domestic                  int64
Beat                      int64
District                float64
community_area            int64
FBI Code                 object
year                      int64
Latitude                float64
Longitude               float64
month                     int64
day_of_week               int64
hour                      int64
is_weekend                int64
community_name           object
per_capita_income       float64
unemployment_rate       float64
no_highschool_pct       float64
date                      int64
TMAX                      int64
TMIN                      int64
PRCP                    float64
SNOW                    float64
AWND                    float64
temp_avg                float64
is_rain                   int64
is_snow                   int64
temp_cat

In [10]:
df.drop(['ID','Date','service_date'], axis=1, inplace=True)


In [11]:
from sklearn.preprocessing import LabelEncoder

cat_cols = ['Primary Type',
    'Description',
    'Location Description',
    'FBI Code',
    'temp_category',
    'day_type']

le_dict = {}

for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

In [12]:
df.dtypes


Primary Type              int64
Description               int64
Location Description      int64
Arrest                    int64
Domestic                  int64
Beat                      int64
District                float64
community_area            int64
FBI Code                  int64
year                      int64
Latitude                float64
Longitude               float64
month                     int64
day_of_week               int64
hour                      int64
is_weekend                int64
community_name           object
per_capita_income       float64
unemployment_rate       float64
no_highschool_pct       float64
date                      int64
TMAX                      int64
TMIN                      int64
PRCP                    float64
SNOW                    float64
AWND                    float64
temp_avg                float64
is_rain                   int64
is_snow                   int64
temp_category             int64
high_wind                 int64
day_type

In [13]:
df.shape

(7970658, 36)

In [14]:
df.to_csv(r"D:\Data_D\TMU\MRP\1.0.Dataset\Crime_Socio_Economic_Weather_Mobility.csv", index=False)
